In [0]:
import dlt
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-8668954358336479>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import col
      3 from pyspark.sql.types import TimestampType

File /databricks/python_shell/dbruntime/autoreload/discoverability/hook.py:72, in AutoreloadDiscoverabilityHook.pre_run_cell.<locals>.patched_import(name, *args, **kwargs)
     66 if not self._should_hint and (
     67     (module := sys.modules.get(absolute_name)) is not None and
     68     (fname := get_allowed_file_name_or_none(module)) is not None and
     69     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     70         absolute_name, float("inf")) and not self._should_hint):
     71     self._should_hint = True
---> 72 module = self._original_builtins_import(name, *args, **kwargs)
     73 if (fname := fname or get_allowed_file_name_or_none(module)) 

In [0]:
@dlt.view(
    name="electroniz_bronze_currency_view",
    comment="Streaming view from bronze currency with selected rates and transformed timestamp"
)
def electroniz_bronze_currency_view():
    return (
        spark.readStream
        .table("electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_currency")
        .select(
            col("base"),
            col("date"),
            col("rates.EUR").alias("EUR"),
            col("rates.USD").alias("USD"),
            col("rates.CAD").alias("CAD"),
            col("rates.INR").alias("INR"),
            col("timestamp").cast(TimestampType()).alias("event_time"),
            col("success")
        )
    )

In [0]:
dlt.create_streaming_table(
    name="electroniz_silver_currency",
    comment="Silver currency table with merged currency rates",
    table_properties={
        "quality": "silver"
    }
)

dlt.apply_changes(
    target="electroniz_silver_currency",
    source="electroniz_bronze_currency_view",
    keys=["date", "base"],
    sequence_by=col("event_time"),
    stored_as_scd_type=1
)